
<div align="center">
  <h1></h1>
  <h1>🦸 Build your Character GPT</h1>
</div>

Tools Required:
*  [PyTorch](https://pytorch.org/)

Steps
 * 🔮 Reading and Exploring Data
 * 🔮 Data Tokenization
 * 🔮 Data Splitting
 * 🔮 Building the DataLoader
 * 🔮 Constructing the GPT Model
 * 🔮 Training and Testing the Model


In [1]:
# install required libraries
!pip install -q torch

# 🔮 Reading and Exploring Data

To train any deep learning model, we need a dataset. For this exercise, we’ll use the **Tiny Shakespeare dataset**, which contains 40,000 lines from various Shakespeare plays. This dataset was featured in Andrej Karpathy's [YouTube tutorial](https://www.youtube.com/watch?v=kCc8FmEb1nY).

In [2]:
# download dataset file
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2024-11-14 10:50:50--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.06s   

2024-11-14 10:50:51 (17.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
# read the data file
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
# get some info about dataset size
print("length of dataset in lines: ",len(text.split("\n")))

length of dataset in lines:  40001


In [5]:
# since we are interested in the dataset size,
# let's calculate the total number of characters in the dataset.
print("Total number of characters in the dataset:", len(text))

Total number of characters in the dataset: 1115394


The dataset contains over one million characters.

In [6]:
# show the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



For the model we are building, we need to calculate the total number of characters that our model can process. This can be determined by extracting all the unique characters from the input text.

In [ ]:
# get all the unique characters in the dataset text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Unique Chars: ",''.join(chars))
print("Length of the unique text", vocab_size)

Unique Chars:  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Length of the unique text 65


# 🔮 Data Tokenization

LLM models can not directly work with characters or strings; instead, they work more efficiently with numbers.

To enable this, we need to assign a unique integer ID to each character in the dataset, which will be used during training and inference. This process is called tokenization.

In [ ]:
# display the unique characters in the dataset.
print("Unique Chars: ",''.join(chars))

Unique Chars:  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
# generate the tokenizer dictionary.
# create a mapping from characters to integers
stringTointeger = { ch:i for i,ch in enumerate(chars) }

# create a mapping from integers to characters
integerToString = { i:ch for i,ch in enumerate(chars) }

To easily convert between a sequence of characters and a sequence of integers, we will create encoding and decoding functions.

In [ ]:
# encoder: take a string, output a list of integers
encode = lambda s: [stringTointeger[c] for c in s]


# decoder: take a list of integers, output a string
decode = lambda l: ''.join([integerToString[i] for i in l])

# test the encoding and decoding functions
encoded_list = encode("Hello GenAI!")
original_test = decode(encoded_list)
print(encoded_list)
print(original_test)

[20, 43, 50, 50, 53, 1, 19, 43, 52, 13, 21, 2]
Hello GenAI!


After building our tokenizer, we will need to tokenize all the characters in our dataset text. For efficient storage and processing, we will use Torch tensors instead of regular Python lists.

In [ ]:
# encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long) # use long data type [int64]
print("dataset size",data.shape)
print("dataset type", data.dtype)

dataset size torch.Size([1115394])
dataset type torch.int64


In [ ]:
# display the first 1000 characters of the text after tokenization.
print(data[:1000])
# for GPT model, the first 1000 characters will appear as follows:

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

# 🔮 Data Splitting

For training the model, we will need a validation dataset to evaluate the model's performance after each training step. To do this, we will split the full dataset into two parts: a training set and a validation set. For simplicity, we will use a 90:10 split ratio.

In [ ]:
# split the data into training and validation sets
# the first 90% will be used for training, and the remaining 10% for validation

# get the length of the training data 90%
n = int(0.9*len(data))

# split based on the training data length
train_data = data[:n]
val_data = data[n:]

# show length of each split
print("training data size", len(train_data))
print("validation data size", len(val_data))

training data size 1003854
validation data size 111540


# 🔮 Building the DataLoader

To train the model, we need to create a dataloader that defines several key characteristics of the model, such as:

- **Context length**: This defines the maximum number of characters the model can process at once. Since the model generates text character by character, the context length determines how much text is considered when predicting the next character.

In [ ]:
# try 8 as context length
context_length = 8

# a block of the data will look like this
train_data[:context_length+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
# the training data features will be like
x = train_data[:context_length]
# the training data labels will be like
y = train_data[1:context_length+1]

# the model will be trined on data like this:
for t in range(context_length):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


- **Batch size**: This defines how many data sequences will be processed independently in parallel, which helps to enhance the training speed by allowing multiple sequences to be processed simultaneously.

In [ ]:
# Batch size: Determines the number of independent sequences processed in parallel during training.
# A larger batch size can speed up training, but may require more memory.
batch_size = 4

In [ ]:
# Update the random seed to ensure reproducibility,
# as we will be using random selection during the process.
torch.manual_seed(1881)

# generate a small batch of data of inputs x and targets y
def get_batch(data):
    # select batch_size random integers between
    # 0 and the length of the data minus the context length,
    #ensuring that each integer represents the starting point of a sequence within the data.
    ix = torch.randint(len(data) - context_length, (batch_size,))

    # get the input and target of the model for all batch sequences
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x, y

xb, yb = get_batch(train_data)
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[56, 43,  1, 52, 53,  1, 51, 53],
        [61, 47, 58, 46,  1, 54, 39, 50],
        [ 0,  0, 31, 43, 56, 60, 39, 52],
        [10,  0, 23, 47, 52, 45,  1, 24]])
targets:
torch.Size([4, 8])
tensor([[43,  1, 52, 53,  1, 51, 53, 56],
        [47, 58, 46,  1, 54, 39, 50, 43],
        [ 0, 31, 43, 56, 60, 39, 52, 58],
        [ 0, 23, 47, 52, 45,  1, 24, 43]])


For training, the model will analyze each sequence along the time dimension, which will allow it to learn and generate text sequentially, step by step, based on the patterns observed in the data.

In [ ]:
for b in range(batch_size): # batch dimension
    for t in range(context_length): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [56] the target: 43
when input is [56, 43] the target: 1
when input is [56, 43, 1] the target: 52
when input is [56, 43, 1, 52] the target: 53
when input is [56, 43, 1, 52, 53] the target: 1
when input is [56, 43, 1, 52, 53, 1] the target: 51
when input is [56, 43, 1, 52, 53, 1, 51] the target: 53
when input is [56, 43, 1, 52, 53, 1, 51, 53] the target: 56
when input is [61] the target: 47
when input is [61, 47] the target: 58
when input is [61, 47, 58] the target: 46
when input is [61, 47, 58, 46] the target: 1
when input is [61, 47, 58, 46, 1] the target: 54
when input is [61, 47, 58, 46, 1, 54] the target: 39
when input is [61, 47, 58, 46, 1, 54, 39] the target: 50
when input is [61, 47, 58, 46, 1, 54, 39, 50] the target: 43
when input is [0] the target: 0
when input is [0, 0] the target: 31
when input is [0, 0, 31] the target: 43
when input is [0, 0, 31, 43] the target: 56
when input is [0, 0, 31, 43, 56] the target: 60
when input is [0, 0, 31, 43, 56, 60] the target:

# 🔮 Constructing the GPT Model

The Generative Pretrained Transformer (GPT) is an autoregressive model of the transformer decoder type, generating text token by token in an autoregressive manner.

In this section, we will build the decoder step by step, following the architecture outlined below:

![image.png](https://www.researchgate.net/publication/373352176/figure/fig1/AS:11431281202501967@1698856108167/GPT-2-model-architecture-The-GPT-2-model-contains-N-Transformer-decoder-blocks-as-shown_W640.jpg)

## Add token Embedding & LM Head Layer

To start, we'll build an embedding layer to capture the context of each character. We'll use PyTorch's default embedding layer for this.  

Additionally, we'll add a final layer to map the embeddings back to the character IDs space, allowing us to convert the model's output back into text.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# determine the embedding size to use for the model.
n_embd = 100

class SimpleGPT(nn.Module):

    def __init__(self):
        super().__init__()
        #  maps token IDs to dense vectors
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # Since the embedding size (n_embed) differs from the vocabulary size,
        # we need an LM head (a linear layer) to map the embeddings back to the vocabulary size for decoding.
        self.lm_head = nn.Linear(n_embd, vocab_size)

    # Forward pass to predict the next token for a batch of input sequences
    def forward(self, idx, targets=None):

        # 'idx' and 'targets' are both tensors of shape (B, T), where:
        # B = batch size, T = sequence length (input tokens)
        tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)

        # pass embeddings through the LM head to predict the next token
        logits = self.lm_head(tok_emb) # (B,T,vocab_size)
        # 'logits' contains the raw predictions (unnormalized scores for each token in the vocabulary).

        if targets is None:
            # Inference mode: no loss computation
            loss = None

        else:
            # Training mode: compute the loss
            # reshape logits and targets to match cross-entropy input format
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # calculate the loss using cross-entropy
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # function to generate a sequence of tokens, given an initial context
    def generate(self, idx, max_new_tokens):
        # 'idx' is a tensor of shape (1, T), representing the input context
        for _ in range(max_new_tokens):
            # trim context to the model's maximum context length
            idx_cond = idx[:, -context_length:] # (1, T)

            # get logits for the next token
            logits, _ = self(idx_cond) # (1, T, vocab_size)

            # focus on the logits for the last time step
            logits = logits[:, -1, :] # becomes (1, vocab_size)

            # convert logits to probabilities using softmax activation function
            probs = F.softmax(logits, dim=-1) # (1, vocab_size)

            # sample the next token from the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (1,1)

            # Append the sampled token to the sequence and loop
            idx = torch.cat((idx, idx_next), dim=1) # (1, T)

        return idx

let's test the model

In [ ]:
# initialize the model.
m = SimpleGPT()

In [ ]:
# pass one input example to the model
print('inputs:')
print(xb)
print('targets:')
print(yb)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

inputs:
tensor([[56, 43,  1, 52, 53,  1, 51, 53],
        [61, 47, 58, 46,  1, 54, 39, 50],
        [ 0,  0, 31, 43, 56, 60, 39, 52],
        [10,  0, 23, 47, 52, 45,  1, 24]])
targets:
tensor([[43,  1, 52, 53,  1, 51, 53, 56],
        [47, 58, 46,  1, 54, 39, 50, 43],
        [ 0, 31, 43, 56, 60, 39, 52, 58],
        [ 0, 23, 47, 52, 45,  1, 24, 43]])
torch.Size([32, 65])
tensor(4.4465, grad_fn=<NllLossBackward0>)


In [ ]:
# retrieve the number of learnable parameters in the model.
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.013065 M parameters


In [ ]:
# Initialize the sequence
initial_seq = torch.zeros((1, 1), dtype=torch.long)
print(initial_seq)

tensor([[0]])


In [ ]:
# generate text using the model architecture.
print(decode(m.generate(idx = initial_seq, max_new_tokens=100)[0].tolist()))


Ytm
 tORKboEM;N,!eiDhTEX&$NidQ,sqCp'TZE;-?zjyfQFXWg;:E;SSgzsvOf-.WLsLrVrNVE'a&JxMWvg
BEvGmQX,FKrWeNt


**Note:** The outputs are random because the model has not been trained yet. This will improve once the model is trained. You can also train each small architecture separately to observe the differences.

## Add positional Embedding & Final Normalization Layer

To represent the position of each token within the input sequence, we need to include a **positional embedding layer**. This layer helps the model understand the order and relative positions of tokens, which is essential for capturing the context and structure of the sequence.

Additionally, we will incorporate a **normalization layer** to ensure consistent scaling and range of the inputs. This layer helps stabilize the training process. Normalization improves the overall convergence and allows the model to train more efficiently, leading to better performance.

In [ ]:
class SimpleGPT(nn.Module):

    def __init__(self):
        super().__init__()
        #  maps token IDs to dense vectors
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        ######################## New Code ########################
        # Create a positional embedding table to encode each position in the sequence
        # Each position is mapped to a vector of size n_embd
        self.position_embedding_table = nn.Embedding(context_length, n_embd)

        # add a final layer normalization to ensure stable training and consistent scaling of the outputs
        self.ln_f = nn.LayerNorm(n_embd)
        ##########################################################

        # Since the embedding size (n_embed) differs from the vocabulary size,
        # we need an LM head (a linear layer) to map the embeddings back to the vocabulary size for decoding.
        self.lm_head = nn.Linear(n_embd, vocab_size)

    # Forward pass to predict the next token for a batch of input sequences
    def forward(self, idx, targets=None):
        ######################## New Code ########################
        # get token position
        B, T = idx.shape
        ##########################################################

        # 'idx' and 'targets' are both tensors of shape (B, T), where:
        # B = batch size, T = sequence length (input tokens)
        tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)

        ######################## New Code ########################
        pos_emb = self.position_embedding_table(torch.arange(T))  # (T, n_embd)
        # add the token embeddings and positional embeddings as per the architecture
        x = tok_emb + pos_emb  # (B, T, n_embd)
        # normalize the output for stable training
        x = self.ln_f(x)  # (B, T, n_embd)
        ##########################################################

        # pass embeddings through the LM head to predict the next token
        logits = self.lm_head(tok_emb) # (B,T,vocab_size)
        # 'logits' contains the raw predictions (unnormalized scores for each token in the vocabulary).

        if targets is None:
            # Inference mode: no loss computation
            loss = None

        else:
            # Training mode: compute the loss
            # reshape logits and targets to match cross-entropy input format
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # calculate the loss using cross-entropy
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # function to generate a sequence of tokens, given an initial context
    def generate(self, idx, max_new_tokens):
        # 'idx' is a tensor of shape (1, T), representing the input context
        for _ in range(max_new_tokens):
            # trim context to the model's maximum context length
            idx_cond = idx[:, -context_length:] # (1, T)

            # get logits for the next token
            logits, _ = self(idx_cond) # (1, T, vocab_size)

            # focus on the logits for the last time step
            logits = logits[:, -1, :] # becomes (1, vocab_size)

            # convert logits to probabilities using softmax activation function
            probs = F.softmax(logits, dim=-1) # (1, vocab_size)

            # sample the next token from the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (1,1)

            # Append the sampled token to the sequence and loop
            idx = torch.cat((idx, idx_next), dim=1) # (1, T)

        return idx


let's test the new model

In [ ]:
# initialize the model.
m = SimpleGPT()

In [ ]:
# pass one input example to the model
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.1900, grad_fn=<NllLossBackward0>)


In [ ]:
# retrieve the number of learnable parameters in the model.
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.014065 M parameters


In [ ]:
# generate text using the model architecture.
print(decode(m.generate(idx = initial_seq, max_new_tokens=100)[0].tolist()))


kECgUOUjIvEi'ANhsCW$OElhqsW'zYnKPcWdZVJa3wddmB&qNqRod:b
;rZQAXb!yxOz;BUIXbwYSR!xlUzg&k
FpT'h,dHBT-dX


## Add Transformer Block

The core component of the GPT model is the **Transformer Block**, which is responsible for learning the relationships and dependencies between tokens in the input sequence. This mechanism, known as self-attention, allows the model to weigh the importance of each token relative to the others, capturing context and meaning even in long sequences. As shown in the architecture below, the GPT model utilizes a **decoder-only** Transformer.

![image.png](https://www.researchgate.net/publication/373352176/figure/fig1/AS:11431281202501967@1698856108167/GPT-2-model-architecture-The-GPT-2-model-contains-N-Transformer-decoder-blocks-as-shown_W640.jpg)

In this structure, each Transformer block consists of multiple layers of self-attention (Multi-Head MAsked Attention) and feedforward neural networks, which work together to process and transform the input sequence.

The Transformer block consists of two main components:
  * **Multi-Head Masked Attention**
  * **Multi-Layer Perceptron (MLP)** (Feedforward Layer)

### Build Multi-Head Attention:

To begin constructing the Transformer layer, we first need to focus on the self-attention mechanism. Self-attention is a crucial component that allows the model to process the entire input sequence simultaneously, considering the relationships between tokens regardless of their position in the sequence.

In self-attention, the representation of each token is transformed based on its interactions with all other tokens in the sequence. This is done by calculating attention scores, which measure how much focus each token should give to the others. The idea is that each token can attend to (or "pay attention to") other tokens to gather relevant contextual information, enabling the model to understand long-range dependencies and relationships.

![](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F966a44ef-fe4b-4d19-900c-543af44dacb2_1614x1042.png)

To calculate the attention between each token, we need to create a self-attention head. This head will compute the attention scores by comparing each token with every other token in the sequence, determining their relevance to each other.

![](https://aiml.com/wp-content/uploads/2023/09/Self-Attention-vs-Multi-headed-Attention.png)

In [ ]:
# update hyperparameters for increased context and improved trainin speed
batch_size = 16
context_length = 32
n_embd = 100

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()

        # using the head size, we will create three linear projections to generate
        # the query, key, and value from the input token embeddings
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # create a lower triangular matrix to maintain causal (autoregressive) token dependencies
        # this prevents future tokens from being attended to by earlier tokens in the sequence
        # Example matrix:
        # [[1., 0., 0., 0.],
        #  [1., 1., 0., 0.],
        #  [1., 1., 1., 0.],
        #  [1., 1., 1., 1.]]
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

        # dropout layer to randomly drop some nodes during training to prevent overfitting
        # setting dropout rate to 0.0 means no nodes will be dropped
        self.dropout = nn.Dropout(0.0)

    # forward pass to compute attention scores between token sequences
    def forward(self, x):
        # x is the input token and position embeddings with shape (B, T, n_embd)
        B,T,C = x.shape

        # get key and query linear projection
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        # compute attention scores using dot product, scaled by n_embd^(-0.5) for stability
        att = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, n_embd) @ (B, n_embd, T) -> (B, T, T)
        # apply a causal mask to the attention scores to prevent attending to future tokens
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)

        # convert attention scores to probabilities using softmax and apply dropout for stability
        att = F.softmax(att, dim=-1) # (B, T, T)
        att = self.dropout(att)

        # perform weighted aggregation of the values to get final attention scores
        v = self.value(x) # (B,T,head_size)
        out = att @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out


Now, we can scale the self-attention mechanism to capture different aspects of the input sequence. To do this, we concatenate multiple attention heads, as shown below:

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention operating in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # create a list of self-attention heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # build a fully connected layer to project the concatenated heads back to the original embedding size
        self.proj = nn.Linear(n_embd, n_embd)

        # dropout layer to prevent overfitting by randomly dropping some nodes during training
        # a dropout rate of 0.0 means no nodes will be dropped
        self.dropout = nn.Dropout(0.0)

    # forward pass for multi-head attention
    def forward(self, x):
        # pass the input sequence through each self-attention head and concatenate the outputs
        out = torch.cat([h(x) for h in self.heads], dim=-1)

        # project the concatenated outputs using the fully connected layer and apply dropout
        out = self.dropout(self.proj(out))

        return out


### Build Multi-Layer Perceptron [MLP] (Feedforward Layer)

Next, we need to build the second part of the Transformer block, which is essential for adding non-linearity to the model. This allows the model to capture more complex relationships within the input sequence.

In [ ]:
class MLP(nn.Module):
    """ a simple feedforward network with a linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        # define a sequential network to introduce non-linearity to the model
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand to a larger size
            # apply a non-linear transformation using GeLU (Gaussian Error Linear Units)
            # GeLU is a smooth activation function that helps with training
            # Reference: https://pytorch.org/docs/stable/generated/torch.nn.GELU.html
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),  # project back to the original embedding size
            nn.Dropout(0.0),  # dropout for regularization, set to 0.0 to avoid dropping for small model size we work on
        )

    # forward pass through the network
    def forward(self, x):
        return self.net(x)


### Construct the Transformer Block

Now that we have implemented the two main components of the Transformer block—**self-attention and the feedforward MLP laye**r—let's move on to constructing the final Transformer block.

In [ ]:
class Block(nn.Module):
    """ Transformer block: self-attention followed by a feedforward network """

    def __init__(self, n_embd, n_head):
        # n_embd: the embedding dimension, n_head: the number of attention heads
        super().__init__()

        # calculate the head size by dividing the embedding dimension by the number of heads
        # this ensures each attention head processes a specific aspect or relationship between tokens
        head_size = n_embd // n_head

        # multi-head self-attention block
        self.sa = MultiHeadAttention(n_head, head_size)

        # feedforward network (MLP) block
        self.ffwd = MLP(n_embd)

        # layer normalization before each block for stable training
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # apply layer normalization to input and pass through multi-head self-attention,
        # then add the input (residual connection) back to the output
        x = x + self.sa(self.ln1(x))

        # normalize the output of the attention block
        x = self.ln2(x)

        # pass through the feedforward network (MLP) and add the input (residual connection) back
        x = x + self.ffwd(x)

        return x


### Assemble the Final GPT Model

Now, we will integrate the Transformer block into the final architecture of the GPT model.

In [ ]:
# Set the hyperparameters
n_head = 4
n_layer = 4

class SimpleGPT(nn.Module):

    def __init__(self):
        super().__init__()
        #  maps token IDs to dense vectors
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # Create a positional embedding table to encode each position in the sequence
        # Each position is mapped to a vector of size n_embd
        self.position_embedding_table = nn.Embedding(context_length, n_embd)

        ######################## New Code ########################
       # add multiple layers of Transformer blocks in a sequential network
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        ##########################################################

        # add a final layer normalization to ensure stable training and consistent scaling of the outputs
        self.ln_f = nn.LayerNorm(n_embd)

        # Since the embedding size (n_embed) differs from the vocabulary size,
        # we need an LM head (a linear layer) to map the embeddings back to the vocabulary size for decoding.
        self.lm_head = nn.Linear(n_embd, vocab_size)

    # Forward pass to predict the next token for a batch of input sequences
    def forward(self, idx, targets=None):
        # get token position
        B, T = idx.shape

        # 'idx' and 'targets' are both tensors of shape (B, T), where:
        # B = batch size, T = sequence length (input tokens)
        tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)

        pos_emb = self.position_embedding_table(torch.arange(T))  # (T, n_embd)
        # add the token embeddings and positional embeddings as per the architecture
        x = tok_emb + pos_emb  # (B, T, n_embd)

        ######################## New Code ########################
        # pass the embeddings to the transformer block
        x = self.blocks(x) # (B,T,n_embd)
        ##########################################################

        # normalize the output for stable training
        x = self.ln_f(x)  # (B, T, n_embd)

        # pass embeddings through the LM head to predict the next token
        logits = self.lm_head(tok_emb) # (B,T,vocab_size)
        # 'logits' contains the raw predictions (unnormalized scores for each token in the vocabulary).

        if targets is None:
            # Inference mode: no loss computation
            loss = None

        else:
            # Training mode: compute the loss
            # reshape logits and targets to match cross-entropy input format
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # calculate the loss using cross-entropy
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # function to generate a sequence of tokens, given an initial context
    def generate(self, idx, max_new_tokens):
        # 'idx' is a tensor of shape (1, T), representing the input context
        for _ in range(max_new_tokens):
            # trim context to the model's maximum context length
            idx_cond = idx[:, -context_length:] # (1, T)

            # get logits for the next token
            logits, _ = self(idx_cond) # (1, T, vocab_size)

            # focus on the logits for the last time step
            logits = logits[:, -1, :] # becomes (1, vocab_size)

            # convert logits to probabilities using softmax activation function
            probs = F.softmax(logits, dim=-1) # (1, vocab_size)

            # sample the next token from the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (1,1)

            # Append the sampled token to the sequence and loop
            idx = torch.cat((idx, idx_next), dim=1) # (1, T)

        return idx

Let's test our GPT model! 🙂

In [ ]:
# initialize the model.
m = SimpleGPT()

In [ ]:
# pass one input example to the model
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.1400, grad_fn=<NllLossBackward0>)


In [ ]:
# retrieve the number of learnable parameters in the model.
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.500465 M parameters


In [ ]:
# generate text using the model architecture.
print(decode(m.generate(idx = initial_seq, max_new_tokens=100)[0].tolist()))


pZ3lpU'
WrXUr3$EycAPW?u'$DT'sjJdxw3kkd:m,lFywvXM$
YNH- HR'uBvBSoZ
heOdosZXBh'sQXghOMPpCAe;MQVXglBvu,


# 🔮 Training and Testing the Model

After building the model architecture, we need to train the model using our dataset splits (training and validation sets).

In [ ]:
# set learning rate for training
learning_rate = 1e-3

# For training, we will use the AdamW optimizer.
# The optimizer adjusts the model's parameters during training
# to minimize the loss function.
# Reference: https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [ ]:
# define the training hyperparameters.
max_iters = 5000        # total number of iterations to train the model
eval_interval = 500     # interval (in iterations) at which to evaluate model performance
eval_iters = 100        # number of iterations to run the evaluation

Before training, we need to implement a function to calculate the training and validation loss of the model.

In [ ]:
# build the loss estimation function to calculate the loss
@torch.no_grad()
def estimate_loss():
    out = {}
    # set the model to evaluation mode.
    m.eval()

    # training loss
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = get_batch(train_data)
        _, loss = m(X, Y)
        losses[k] = loss.item()
        out["train"] = losses.mean()

    # validation loss
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = get_batch(val_data)
        _, loss = m(X, Y)
        losses[k] = loss.item()
        out["val"] = losses.mean()

    # return to training mode
    m.train()
    return out

In [ ]:
# trianing the model
from tqdm import tqdm
for iter in tqdm(range(max_iters)):

    # periodically evaluate the loss on both the training and validation sets.
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"\nstep {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch(train_data)

    # perform the forward pass.
    logits, loss = m(xb, yb)

    # reset the optimizer's gradients before the backpropagation step.
    optimizer.zero_grad(set_to_none=True)
    # calculate the gradients for each parameter in the model.
    loss.backward()
    # update the model parameters based on the calculated gradients.
    optimizer.step()

  0%|          | 5/5000 [00:04<1:02:04,  1.34it/s]


step 0: train loss 4.3745, val loss 4.3732


 10%|█         | 504/5000 [00:26<28:38,  2.62it/s]


step 500: train loss 2.5090, val loss 2.5282


 20%|██        | 1004/5000 [00:48<30:19,  2.20it/s]

...


step 4000: train loss 2.4538, val loss 2.4936


 90%|█████████ | 4506/5000 [03:19<02:14,  3.67it/s]


step 4500: train loss 2.4613, val loss 2.4894


100%|██████████| 5000/5000 [03:41<00:00, 22.58it/s]


step 4999: train loss 2.4653, val loss 2.4815


Test the model after training

In [ ]:
# generate text after training
print(decode(m.generate(idx = initial_seq, max_new_tokens=100)[0].tolist()))


I l ameneaist:

Bucherithid scetowhan?


An
Ane yomsenc kpouarrcand swe'd.
MENCl:

Wast ssthonta as 


Build a generation method for text completion

In [ ]:
def generate_gpt(input_text):
    # encode the text
    start_sequence = torch.tensor([encode(input_text)], dtype=torch.long)
    # get model completion
    generation = m.generate(start_sequence, max_new_tokens=100)
    # decode the output ids
    generated_text = decode(generation[0].tolist())
    print(generated_text)


In [ ]:
# run the model
generate_gpt("What would you have, friend?")

What would you have, friend? me.
Astirshe

Soun pr y il cht t shimegem.
Ather, e mecher,
ARGoare furieee cheerou larthult theruc
